# Initialize, sample, and review SpecMine data

Run the cells from top to bottom. Existing metadata and feature parquet files in `notebooks/data` are read locally; if either is missing, the notebook downloads the parquet files into that cache first. It then downloads or reuses a random sample of `SPEC_SAMPLE_SIZE` raw spec files (300 by default). The sample contains one unique spec from each unique repository; failed downloads are replaced by other repositories. Next, the notebook chooses one random repository from that sample that is not already in either label file. The final cell displays its spec Markdown, precomputed features, and selected metadata attributes. Set `SEED = None` for a new random selection each run, or choose an integer seed to reproduce a sample.

In [12]:
import html
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

from specmine.data import (
    download_dataset_parquets,
    download_random_spec_sample,
    load_data,
    load_feature_data,
)
from specmine.labeling import match_review_features, select_review_spec
from specmine.labels import read_labels_csv

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))

SEED = None
SPEC_SAMPLE_SIZE = 1
LOCAL_DATA_ONLY = True
DATA_DIR = project_root / "notebooks" / "data"
REAL_LABELS_CSV = project_root / "examples" / "real_labels.csv"
FAKE_LABELS_CSV = project_root / "examples" / "fake_labels.csv"

print(f"Data directory: {DATA_DIR}")
print(f"Requested distinct spec/repository count: {SPEC_SAMPLE_SIZE}")

Data directory: /Users/austinburt/Documents/Code/PythonEnv/Python/CS440/FinalProject-CS440/notebooks/data
Requested distinct spec/repository count: 1


In [13]:
if SPEC_SAMPLE_SIZE < 1:
    raise ValueError("SPEC_SAMPLE_SIZE must be a positive integer.")

parquet_names = ("spec_files.parquet", "spec_content_features.parquet")
parquet_files = {
    name: next(
        (
            path
            for path in (DATA_DIR / "data" / name, DATA_DIR / name)
            if path.is_file()
        ),
        None,
    )
    for name in parquet_names
}
if any(path is None for path in parquet_files.values()):
    print("One or more parquet files are missing; downloading the dataset cache.")
    parquet_files = download_dataset_parquets(DATA_DIR)
else:
    print("Using the parquet files already in the data directory.")
for name, path in parquet_files.items():
    print(f"Cached {name}: {path} ({path.stat().st_size / (1024 * 1024):.1f} MiB)")

metadata = load_data(local_data_only=LOCAL_DATA_ONLY, data_dir=DATA_DIR)
if metadata is None or metadata.empty:
    raise RuntimeError("The local SpecMine metadata table is empty or unavailable.")

sampled_specs = download_random_spec_sample(
    metadata,
    SPEC_SAMPLE_SIZE,
    seed=SEED,
    data_dir=DATA_DIR,
)
if len(sampled_specs) != SPEC_SAMPLE_SIZE:
    raise RuntimeError(
        f"Expected {SPEC_SAMPLE_SIZE} downloaded specs, got {len(sampled_specs)}."
    )
if sampled_specs["file_url_sha16"].nunique() != SPEC_SAMPLE_SIZE:
    raise RuntimeError("The sample contains duplicate spec IDs.")
if sampled_specs["repo_id"].nunique() != SPEC_SAMPLE_SIZE:
    raise RuntimeError("The sample contains duplicate repository IDs.")

manifest_path = DATA_DIR / "spec_sample.csv"
sampled_specs.to_csv(manifest_path, index=False)
print(
    f"Ready: {len(sampled_specs)} unique spec files from "
    f"{sampled_specs['repo_id'].nunique()} unique repositories."
)
print(f"Sample manifest: {manifest_path}")

Using the parquet files already in the data directory.
Cached spec_files.parquet: /Users/austinburt/Documents/Code/PythonEnv/Python/CS440/FinalProject-CS440/notebooks/data/data/spec_files.parquet (644.3 MiB)
Cached spec_content_features.parquet: /Users/austinburt/Documents/Code/PythonEnv/Python/CS440/FinalProject-CS440/notebooks/data/data/spec_content_features.parquet (10.3 MiB)
Processed 1/1 files
Downloaded or reused 1/1 files
Ready: 1 unique spec files from 1 unique repositories.
Sample manifest: /Users/austinburt/Documents/Code/PythonEnv/Python/CS440/FinalProject-CS440/notebooks/data/spec_sample.csv


## Choose one repository to review

The next cell chooses one random repository from the downloaded sample. It checks the real and fake label files for repository overlap and excludes repositories that already appear in either file. It selects one spec from the chosen repository and keeps its IDs in `review_spec` for the display cell below.

In [21]:
real_labels = read_labels_csv(REAL_LABELS_CSV)
fake_labels = read_labels_csv(FAKE_LABELS_CSV)
review_spec = select_review_spec(
    sampled_specs,
    real_labels,
    fake_labels,
    seed=SEED,
    label_metadata=metadata,
)
if len(review_spec) != 1:
    raise RuntimeError("Repository selection must contain exactly one spec.")

review_row = review_spec.iloc[0]
print(f"Selected repository: {review_row['repo_name']}")
print(f"repo_id: {review_row['repo_id']}")
print(f"spec_id: {review_row['file_url_sha16']}")
print(f"Spec file: {review_row['file_path']}")

Selected repository: nrajesh/vaulted.money
repo_id: 1051228744
spec_id: 751356db45475204
Spec file: specs/030-public-homepage-trust-onboarding/spec.md


## Spec file, features, and metadata

This cell uses the repository/spec chosen above; no IDs need to be copied manually. It displays the selected file's exact precomputed feature row and useful repository/file attributes. Commit history is intentionally not displayed or used.

In [15]:
feature_rows = load_feature_data(
    review_spec[["file_url_sha16"]],
    local_data_only=True,
    data_dir=DATA_DIR,
)
review_features = match_review_features(review_spec, feature_rows)

display_columns = [
    "repo_id",
    "repo_name",
    "repo_description",
    "repo_language",
    "repo_stars",
    "repo_topics",
    "file_url_sha16",
    "file_path",
    "file_name",
    "file_sha",
    "file_content_sha",
    "file_content_bytes",
    "content_fetch_status",
    "spec_tool",
    "spec_role",
]
available_columns = [column for column in display_columns if column in review_spec]
display(HTML("<h3>Repository and spec attributes</h3>"))
display(review_spec[available_columns].reset_index(drop=True))
display(HTML("<h3>Precomputed content features</h3>"))
display(review_features)

spec_path = Path(review_row.local_path)
if not spec_path.is_file():
    raise FileNotFoundError(f"Selected spec file is missing: {spec_path}")
spec_markdown = spec_path.read_text(encoding="utf-8")
display(HTML("<h3>Spec Markdown</h3>"))
display(HTML("<pre>" + html.escape(spec_markdown) + "</pre>"))

print(
    "To record a manual review, add repo_id,spec_id,label,confidence to "
    "examples/real_labels.csv. Use label 0, 1, or blank if ambiguous; "
    "confidence is 0-100 for a nonblank label."
)

,repo_id,repo_name,repo_description,repo_language,repo_stars,repo_topics,file_url_sha16,file_path,file_name,file_sha,file_content_sha,file_content_bytes,content_fetch_status,spec_tool,spec_role
0,1051228744,nrajesh/vaulted.money,"A privacy-focused, local-first personal financ...",TypeScript,9,"account-tracking, budget, cashflow, dexiejs, e...",751356db45475204,specs/030-public-homepage-trust-onboarding/spe...,spec.md,36c4c97c583c90db8dc49b273dd6b25dc8333c97,fc696830e146903d71bb4386047e2cebaba0f5458ab4e3...,3908.0,done,speckit,feature


,file_url_sha16,n_bytes,n_lines,n_words,n_headings,max_heading_depth,n_code_fences,n_mermaid,n_tables,n_links,...,has_edge_cases,has_open_questions,has_dependencies,has_purpose,has_mandatory_marker,is_tiny,has_unfilled_placeholder,has_lorem,lang,content_family
0,751356db45475204,3908,53,619,7,3,0,0,0,0,...,0,0,0,0,0,0,0,0,en,unknown


To record a manual review, add repo_id,spec_id,label,confidence to examples/real_labels.csv. Use label 0, 1, or blank if ambiguous; confidence is 0-100 for a nonblank label.
